# Notebook 00 — Data Quality & Completeness Check

Run this notebook first before any analysis.  
It verifies that `benchmark_results.csv` is structurally sound:
- Correct column schema
- Expected model × query × run-type × run-count combinations
- No unexpected `Status` values
- Plausible numeric ranges
- Documents the known NaN pattern (Q4 has no DuckDB profiling output)


In [1]:
import pandas as pd
import numpy as np

CSV_PATH = '../results/benchmark_results.csv'

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')
df.head(3)

Loaded 2,760 rows, 10 columns


,SF,Model,Query,RunType,RunNumber,WallClock_ms,DuckDB_Total_ms,DuckDB_Planning_ms,PeakRAM_KB,Status
0,10,A1,Q1,COLD,1,788.44,678.238,0.0,88236,OK
1,10,A1,Q1,COLD,2,714.37,625.250,0.0,87824,OK
2,10,A1,Q1,COLD,3,713.24,644.538,0.0,89008,OK


## 1  Schema validation

In [2]:
EXPECTED_COLS = {
    'SF': 'int64',
    'Model': 'object',
    'Query': 'object',
    'RunType': 'object',
    'RunNumber': 'int64',
    'WallClock_ms': 'float64',
    'DuckDB_Total_ms': 'float64',
    'DuckDB_Planning_ms': 'float64',
    'PeakRAM_KB': 'int64',
    'Status': 'object',
}

missing = [c for c in EXPECTED_COLS if c not in df.columns]
extra   = [c for c in df.columns if c not in EXPECTED_COLS]
print('Missing columns :', missing or 'none')
print('Unexpected columns:', extra or 'none')
print()
print(df.dtypes)

Missing columns : none
Unexpected columns: none

SF                      int64
Model                  object
Query                  object
RunType                object
RunNumber               int64
WallClock_ms          float64
DuckDB_Total_ms       float64
DuckDB_Planning_ms    float64
PeakRAM_KB              int64
Status                 object
dtype: object


## 2  Unique value catalogue

In [3]:
for col in ['SF', 'Model', 'Query', 'RunType', 'Status']:
    print(f'{col:12s}: {sorted(df[col].unique())}')

SF          : [np.int64(1), np.int64(10)]
Model       : ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'B1', 'B2', 'B3', 'B4', 'C1']
Query       : ['Q1', 'Q2a', 'Q2b', 'Q2c', 'Q3_B1', 'Q3_B2', 'Q3_B3', 'Q3_B4_day', 'Q3_B4_month', 'Q3_Both', 'Q3_Month', 'Q3_Year', 'Q4', 'Q5', 'Q6', 'Q7']
RunType     : ['COLD', 'WARM']
Status      : ['OK']


## 3  Run-count completeness matrix

In [4]:
EXPECTED_COLD = 20
EXPECTED_WARM = 3

counts = (
    df.groupby(['SF', 'Model', 'Query', 'RunType'])
    .size()
    .reset_index(name='n')
)

def expected_n(run_type):
    return EXPECTED_COLD if run_type == 'COLD' else EXPECTED_WARM

counts['expected'] = counts['RunType'].map(
    {'COLD': EXPECTED_COLD, 'WARM': EXPECTED_WARM}
)
counts['ok'] = counts['n'] == counts['expected']

problems = counts[~counts['ok']]
if problems.empty:
    print('✓  All combinations have the expected run count.')
else:
    print('✗  Unexpected run counts:')
    print(problems.to_string(index=False))

print(f'\nTotal combinations checked: {len(counts)}')
counts.head(10)

✓  All combinations have the expected run count.

Total combinations checked: 240


,SF,Model,Query,RunType,n,expected,ok
0,1,A1,Q1,COLD,20,20,True
1,1,A1,Q1,WARM,3,3,True
2,1,A1,Q2a,COLD,20,20,True
3,1,A1,Q2a,WARM,3,3,True
4,1,A1,Q2b,COLD,20,20,True
5,1,A1,Q2b,WARM,3,3,True
6,1,A1,Q2c,COLD,20,20,True
7,1,A1,Q2c,WARM,3,3,True
8,1,A1,Q4,COLD,20,20,True
9,1,A1,Q4,WARM,3,3,True


## 4  Status check

In [5]:
bad_status = df[df['Status'] != 'OK']
if bad_status.empty:
    print('✓  All rows have Status=OK.')
else:
    print(f'✗  {len(bad_status)} rows with Status != OK:')
    print(bad_status.groupby(['SF','Model','Query','RunType','Status']).size())

✓  All rows have Status=OK.


## 5  NaN audit

In [6]:
print('NaN counts per column:')
print(df.isna().sum())
print()

# Known pattern: DuckDB profiling is not emitted for COUNT(*) (Q4)
nan_rows = df[df['DuckDB_Total_ms'].isna()]
print('Rows with NaN DuckDB_Total_ms:')
print(nan_rows.groupby(['Model', 'Query']).size().reset_index(name='n'))
print()
print('NOTE: Q4 (COUNT*) does not produce JSON profiling output in DuckDB v1.5.0.')
print('      WallClock_ms and PeakRAM_KB are still recorded and are valid.')

NaN counts per column:
SF                     0
Model                  0
Query                  0
RunType                0
RunNumber              0
WallClock_ms           0
DuckDB_Total_ms       46
DuckDB_Planning_ms    46
PeakRAM_KB             0
Status                 0
dtype: int64

Rows with NaN DuckDB_Total_ms:
  Model Query   n
0    A1    Q4  46

NOTE: Q4 (COUNT*) does not produce JSON profiling output in DuckDB v1.5.0.
      WallClock_ms and PeakRAM_KB are still recorded and are valid.


## 6  Numeric range sanity checks

In [7]:
cold = df[df['RunType'] == 'COLD']

print('WallClock_ms summary (cold runs):')
print(cold['WallClock_ms'].describe().round(1))
print()
print('PeakRAM_KB summary (cold runs):')
print(cold['PeakRAM_KB'].describe().round(0))

# Flag anything suspiciously low
low = cold[cold['WallClock_ms'] < 10]
if not low.empty:
    print(f'\n⚠  {len(low)} cold-run rows with WallClock_ms < 10 ms:')
    print(low[['SF','Model','Query','RunNumber','WallClock_ms']].to_string(index=False))
else:
    print('\n✓  No suspiciously low wall-clock values.')

WallClock_ms summary (cold runs):
count      2400.0
mean      22437.2
std       74915.9
min          52.0
25%         116.8
50%         753.6
75%        1674.0
max      390169.0
Name: WallClock_ms, dtype: float64

PeakRAM_KB summary (cold runs):
count        2400.0
mean      1156996.0
std       3388794.0
min         30216.0
25%         54405.0
50%         82840.0
75%        207584.0
max      14336640.0
Name: PeakRAM_KB, dtype: float64

✓  No suspiciously low wall-clock values.


## 7  Model availability per scale factor

A2 only exists at SF-10 (a single 7.5 GB file would be identical to A1 at SF-1).

In [8]:
for sf in sorted(df['SF'].unique()):
    models = sorted(df[df['SF'] == sf]['Model'].unique())
    print(f'SF-{sf}: {models}')

SF-1: ['A1', 'A3', 'A4', 'A5', 'A6', 'A7', 'B1', 'B2', 'B3', 'B4', 'C1']
SF-10: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'B1', 'B2', 'B3', 'B4', 'C1']


## 8  Summary report

In [9]:
n_sfs     = df['SF'].nunique()
n_models  = df['Model'].nunique()
n_queries = df['Query'].nunique()
n_cold    = len(df[df['RunType']=='COLD'])
n_warm    = len(df[df['RunType']=='WARM'])

print('=== Benchmark dataset summary ===')
print(f'  Scale factors : {n_sfs}')
print(f'  Models        : {n_models}')
print(f'  Query types   : {n_queries}')
print(f'  Cold-cache rows: {n_cold:,}')
print(f'  Warm-cache rows: {n_warm:,}')
print(f'  Total rows     : {len(df):,}')
print()
print('Data quality: PASS' if problems.empty and bad_status.empty else 'Data quality: ISSUES FOUND — review cells above')

=== Benchmark dataset summary ===
  Scale factors : 2
  Models        : 12
  Query types   : 16
  Cold-cache rows: 2,400
  Warm-cache rows: 360
  Total rows     : 2,760

Data quality: PASS
